# Exercise 15: Power analyses

This  assignment is designed to give you practice with Monte Carlo methods to conduct power analyses via simulation. You won't need to load in any data for this homework. We will, however, be using parts of the homework from last week.

---
## 1. Simulating data (1 point)


Pull your `simulate_data()` function from your last homework and add it below.

As a reminder, this function simulates the relationship between age, word reading experience, and reading comprehension skill.

`c` is reading comprehension, and `x` is word reading experience.

In [15]:
sample_size = 100 # How many children in data set?
age_lo = 80     # minimum age, in months
age_hi = 200    # maximum age, in months
beta_xa = 0.5   # amount by which experience changes for increase of one month in age
beta_x0 = -5    # amount of experience when age = 0 (not interpretable, since minimum age for this data is 80 months)
sd_x = 50       # standard dev of gaussian noise term, epsilon_x
beta_ca = 0.8   # amount that comprehension score improves for every increase of one unit in age
beta_cx = 3     # amount that comprehension score improves for every increase of one unit in reading experience
beta_c0 = 10    # comprehension score when reading experience is 0.
sd_c = 85      # standard dev of gaussian noise term, epsilon_c

simulate_data <- function(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c) {
    age = runif(sample_size, age_lo, age_hi)
    x = beta_xa*age + beta_x0 + rnorm(sample_size, 0, sd_x)
    c = beta_ca*age + beta_cx*x + beta_c0 + rnorm(sample_size, 0, sd_c)
    return(data.frame(age=age,x=x,c=c))
}

dat <- simulate_data(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)
head(dat)

,age,x,c
,<dbl>,<dbl>,<dbl>
1,139.58948,45.692835,212.64558
2,169.42987,8.643142,69.88375
3,175.72387,50.001702,161.80424
4,104.63147,93.495394,298.77611
5,93.65171,17.431299,149.16740
6,120.10699,63.162038,228.72397


---
## 2. `run_analysis()` function (2 points)

Last week, we looked at whether word reading experience(`x`) mediated the relation between `age` and reading comprehension (`c`).

Now we're going to use our `simulate_data()` function to conduct a power analysis. The goal is to determine how many participants we would need in order to detect both the mediated and the direct effects in this data.

*Note: We're going to pretend for the sake of simplicity that we don't have any control over the ages of the children we get (so ages are generated using `runif(sample_size, age_lo, age_hi)`, although of course this would be an unusual situation in reality.*

First, write a function, `run_analysis()`, that takes in simulated data, runs **your mediation from last week**, and returns a vector containing the ACME and ADE estimates and p-values (these are the `d0`, `d0.p`, `z0`, and `z0.p` features of the mediated model object, e.g., `fitMed$d0.p`). Print this function's output for the data we simulated previously.

In [16]:
library(mediation)

run_analysis <- function(df){
    mod_1 <- lm(x ~ age, data = df)
    mod_2 <- lm(c ~ age + x, data = df)
    med_mod <- mediate(mod_1, mod_2, treat = "age", mediator = "x")
    output <- rbind(med_mod$d0, med_mod$d0.p, med_mod$z0, med_mod$z0.p)
    return(output)
}

run_analysis(dat)

1.9118077
0.0000000
0.6423195
0.0100000


---
## 3. `repeat_analysis()` function (3 points)

Next fill in the function `repeat_analysis()` below so that it simulates and analyzes data `num_simulations` times. Store the outputs from each simulation in the `simouts` matrix. Calculate and return the coverage across all the simulations run for both ACME and ADE.

In [17]:
repeat_analysis <- function(num_simulations, alpha, sample_size, age_lo, age_hi,
        beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c) {
    # Initialize simouts matrix for storing each output from run_analysis()
    simouts <- matrix(rep(NA, num_simulations*4), nrow=num_simulations, ncol=4)

    # Start simulating
    for (i in 1:num_simulations) {
        df <- simulate_data(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)
        simouts[i,] <- run_analysis(df)
    }

    # Calculate coverage for both ACME and ADE estimates using p-values in simouts
    ACME_cov = mean(simouts[,2] <= alpha)
    ADE_cov =  mean(simouts[,4] <= alpha)

    return(list(ACME_cov = ACME_cov, ADE_cov = ADE_cov))
}

Now run the `repeat_analysis()` function using the same parameter settings as above, for 10 simulations, with an alpha criterion of 0.01.

In [18]:
repeat_analysis(10, 0.01, sample_size, age_lo, age_hi, 
        beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)

$ACME_cov
[1] 0.7

$ADE_cov
[1] 0.8

---
## 4. Testing different sample sizes (2 points)

Finally, do the same thing (10 simulations, alpha criterion of 0.01) but for 5 different sample sizes: 50, 75, 100, 125, 150. You can do this using `map` (as in the tutorial), or a simple `for` loop, or by calculating each individually. Up to you! This should take around 3 minutes to run.

In [19]:
library(tidyverse)

df <- expand.grid(sample_size = seq(50, 150, 25))
results <- df %>%
        mutate(power = map(sample_size, ~repeat_analysis(10, 0.01, .x, age_lo, age_hi, 
        beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)))

Print your results.

In [21]:
print(results) # power[1] acme - mediated, power[2] - direct effect

  sample_size    power
1          50 0.7, 0.5
2          75 0.9, 0.2
3         100 0.8, 0.6
4         125 1.0, 0.7
5         150 0.9, 0.8


## 5. Reflection (2 pts)

If this were a real power analysis, we'd want to run more simulations per sample size (to get a more precise estimate of power) and we may also want to test out some other values of the parameters we used to simulate our data. However, what would you conclude just based on the results above?

> Overall, as the sample size increases, the power of the study also increases. There is a strong mediating effect, thus we require a smaller sample size to detect it. The direct effect is weaker, and needs a larger sample size to be detected, e.g., minimum n = 150 and ~80% probability of being detected versus n = 125 with 100% probability for the mediating effect.

Given how we generated the data, why was the direct effect harder to detect than the mediated effect?
> The direct effect was harder to detect than the mediated effect because of the shared variance between the mediator variable and the predictor variable. Since the variance contributed by the mediator to X is always accounted for, the direct effect of X is weaker due to less unique variance. 

**DUE:** 11:59pm EST, March 31, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> I did not collaborate with any of my classmates on this assignment. 

**GenAI Utilization** Did you utilize any generative AI tools on this assignment? If so, please list the item and the paste respective prompt you used.

> I did not utilize any generative AI tools on this assignment. Though I referenced the tutorials on the class jupyter notebook. 